In [3]:
import os
from langchain_cerebras import ChatCerebras
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
from pymongo import MongoClient
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException
from langchain.agents import create_agent

load_dotenv()

CEREBRAS_API_KEY=os.getenv("CEREBRAS_API_KEY")
BREVO_API_KEY=os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")


In [4]:
DB_NAME = "soulsync"
COLLECTION_NAME="users"
SENDER_NAME="SoulSync"
SENDER_MAIL="lokeshvijayraina@gmail.com"


In [5]:
llm_gpt=ChatCerebras(model="gpt-oss-120b",api_key=CEREBRAS_API_KEY)
client= MongoClient(MONGODB_URI)
users_collection=client[DB_NAME][COLLECTION_NAME]


In [6]:
class EmailTemplate(BaseModel):
    subject_template:str
    body_template:str
    reason:str

class EmailDraft(BaseModel):
    receiver_mail:str
    subject:str
    body:str

class DispatchedUsers(BaseModel):
    totalUsers:str
    sent:str
    failed:str

In [10]:
@tool
def find_users(
    filters: dict = None,
    sort_by: str = None,
    ascending: bool = False,
    limit: int = 10,
):
    """
    Query SoulSync users with optional filters, sorting, and limit.

    Available fields:
    - name
    - email
    - totalListeningTime
    - updatedAt (last active)
    - createdAt (joined date)
    """

    query = filters or {}

    targeted_users = users_collection.find(
        query,
        {
            "_id": 0,
            "name": 1,
            "email": 1,
            "totalListeningTime": 1,
            "updatedAt": 1,
            "createdAt": 1,
        },
    )

    if sort_by:
        targeted_users = targeted_users.sort(
            sort_by,
            -1 if not ascending else 1,
        )

    targeted_users = targeted_users.limit(limit)

    return list(targeted_users)

In [12]:
find_users.invoke({"filters": {}, "sort_by": "createdAt", "ascending": True, "limit": 10})


[{'createdAt': datetime.datetime(2026, 3, 4, 8, 44, 49, 307000),
  'email': 'lokeshvijayraina@gmail.com',
  'name': 'Loki',
  'totalListeningTime': 471426,
  'updatedAt': datetime.datetime(2026, 7, 5, 18, 31, 24, 628000)},
 {'createdAt': datetime.datetime(2026, 3, 4, 12, 54, 28, 727000),
  'email': 'suvasuvakovan@gmail.com',
  'name': 'ApeX',
  'totalListeningTime': 3453,
  'updatedAt': datetime.datetime(2026, 3, 4, 13, 47, 37, 160000)},
 {'createdAt': datetime.datetime(2026, 3, 4, 12, 54, 53, 897000),
  'email': 'mohan24raj18@gmail.com',
  'name': 'Mohan Raj',
  'totalListeningTime': 599,
  'updatedAt': datetime.datetime(2026, 3, 6, 14, 3, 4, 272000)},
 {'createdAt': datetime.datetime(2026, 3, 4, 13, 1, 37, 523000),
  'email': 't6863126@gmail.com',
  'name': 'The hacker',
  'totalListeningTime': 483,
  'updatedAt': datetime.datetime(2026, 3, 6, 2, 9, 25, 871000)},
 {'createdAt': datetime.datetime(2026, 3, 4, 13, 1, 44, 293000),
  'email': 'reshmisankar9256@gmail.com',
  'name': 'Minio